# 🚀 MVP-B：多產業 RAG 知識庫平台
### 技術展示：FastAPI + LangChain + ChromaDB

這個 Notebook 展示了如何建立一個可切換「金融、醫療、法務」領域的 RAG 系統。

In [ ]:
!pip install langchain langchain-openai chromadb tiktoken python-dotenv

In [ ]:
import os
from google.colab import userdata

# 如果在 Colab 執行，請在 Secrets 設定 OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') if 'google.colab' in str(get_ipython()) else "YOUR_API_KEY"

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.text_splitter import CharacterTextSplitter

class RAGEngine:
    def __init__(self):
        self.embeddings = OpenAIEmbeddings()
        self.llm = ChatOpenAI(model_name="gpt-4-turbo-preview", temperature=0)
        self.vectorstores = {}

    def add_data(self, industry, texts):
        text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
        docs = text_splitter.create_documents(texts)
        self.vectorstores[industry] = Chroma.from_documents(docs, self.embeddings)

    def ask(self, industry, query):
        if industry not in self.vectorstores:
            return "該領域尚未建立知識庫"
        qa = RetrievalQA.from_chain_type(llm=self.llm, chain_type="stuff", retriever=self.vectorstores[industry].as_retriever())
        return qa.invoke({"query": query})["result"]

engine = RAGEngine()
print("RAG 引擎初始化完成")

In [ ]:
# 寫入範例數據
engine.add_data("finance", ["0050 是台灣最知名的市值型 ETF。", "股市投資應注意風險分散。"])
engine.add_data("medical", ["腎病飲食應控制磷、鉀、鈉的攝取。", "長照 2.0 提供居家照顧服務。"])

print("查詢金融領域：", engine.ask("finance", "什麼是 0050？"))
print("查詢醫療領域：", engine.ask("medical", "腎病飲食要注意什麼？"))